# SentinelDesk triage: train on Colab (from VS Code)

**How to connect (VS Code + Google Colab extension):** open this notebook → **Select Kernel** (top right) → **Colab** → **New Colab Server** → pick a **T4 GPU** runtime. Then run the cells top to bottom.

The Colab runtime is a remote machine: it does **not** see your laptop files. So we `git clone` the repo (push your branch first) and send the trained artifacts back at the end.

Rules that still apply here (see `CLAUDE.md`): the split comes only from `split.py`; nothing reads the holdout labels except `eval_triage.py`, and that runs **once, at the very end**.

In [ ]:
# 1. Check the GPU (expect a Tesla T4). If this fails, switch the runtime to a GPU one.
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Get the code
Set `BRANCH` to the branch you pushed. The repo must be reachable from Colab (public, or use a token URL).

In [ ]:
REPO_URL = "https://github.com/wuhong4088/HackathonGDG.git"
BRANCH = "shatru"
WORKDIR = "/content/HackathonGDG"

import os
if os.path.exists(WORKDIR):
    !git -C {WORKDIR} fetch origin {BRANCH} && git -C {WORKDIR} checkout {BRANCH} && git -C {WORKDIR} reset --hard origin/{BRANCH}
else:
    !git clone --branch {BRANCH} --depth 1 {REPO_URL} {WORKDIR}
%cd {WORKDIR}
!git log -1 --oneline

In [ ]:
# 3. Install dependencies (keeps Colab's CUDA torch). scikit-learn must be 1.7.2 so the laptop can load the models.
!pip install -q -r requirements.txt
import sklearn, numpy, torch
print("sklearn", sklearn.__version__, "| numpy", numpy.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available())
assert sklearn.__version__ == "1.7.2", "Restart the runtime (Runtime > Restart) and re-run from step 2"

## 4. Settings
* `EMBED_MODEL`: `intfloat/multilingual-e5-small` (fast, also fine on a laptop CPU) or `BAAI/bge-m3` (stronger, slower; the live demo then embeds each ticket with bge-m3 too).
* Artifacts go to `/content/artifacts` (local disk is fastest); step 9 copies them out.

In [ ]:
EMBED_MODEL = "intfloat/multilingual-e5-small"   # or "BAAI/bge-m3"
ARTIFACTS = "/content/artifacts"
os.environ["SENTINELDESK_ARTIFACTS"] = ARTIFACTS
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# Optional: an HF token avoids Hub rate limits (dataset + model downloads).
# from getpass import getpass; os.environ["HF_TOKEN"] = getpass("HF token (optional): ")

In [ ]:
# 5. Tests + a quick smoke run on 2,000 tickets (~1-2 min). Catches problems before the full run.
!python -m pytest -q tests/test_triage.py
!python scripts/build_split.py --sample 2000 && python scripts/embed.py --sample 2000 --model {EMBED_MODEL} && python scripts/train_triage.py --sample 2000 --model {EMBED_MODEL} | tail -5

## 6. Full training
Expected on a T4: split <1 min, embeddings ~1-2 min (e5-small) / ~5-8 min (bge-m3), training ~3-6 min.

In [ ]:
%%time
!python scripts/build_split.py

In [ ]:
%%time
!python scripts/embed.py --model {EMBED_MODEL}

In [ ]:
%%time
# Trains 0, 1, 5 on train; ensemble on half A; temperature + thresholds on half B. Prints half-B metrics.
!python scripts/train_triage.py --model {EMBED_MODEL}

In [ ]:
# 7. Sanity check: load the trained Triage and predict two live tickets (English + German, one without subject).
import sys, json; sys.path.insert(0, WORKDIR)
from sentineldesk.triage.predict import Triage
triage = Triage.load(ARTIFACTS)
for subj, body in [("VPN outage", "Since 8am nobody in the clinic can reach the EMR system, all users affected."),
                   (None, "Guten Tag, ich habe eine Frage zu meiner letzten Rechnung. Können Sie mir eine Kopie schicken?")]:
    r = triage.predict(subj, body)
    print(json.dumps({k: r[k] for k in ("mode", "critical", "latency_ms")} |
                     {t: (r[t]["label"], r[t]["confidence"]) for t in ("type", "queue", "priority")}, ensure_ascii=False, indent=1))

## 8. Holdout evaluation (ONCE, at the very end)
Only run this when the models are final. Never change keywords, thresholds or hyperparameters after looking at these numbers.

In [ ]:
RUN_HOLDOUT_EVAL = False   # set True only for the final run
if RUN_HOLDOUT_EVAL:
    !python scripts/eval_triage.py
    !mkdir -p {ARTIFACTS}/results && cp -r results/triage_metrics.json results/plots {ARTIFACTS}/results/

## 9. Send the artifacts back to your laptop
The VS Code Colab extension does not support `drive.mount()` or `files.download()` reliably, so try these in order:

1. **Google Drive** (works if mounting succeeds in your extension version)
2. **Private Hugging Face repo** (most reliable; needs an HF *write* token)
3. Fall back to the Colab file browser / "Download" on `/content/sentineldesk_artifacts.zip`

The zip holds `triage/` (all the demo needs, self-contained), `embeddings/` (reusable by the retrieval teammate) and `splits/`.

In [ ]:
# Build the zip (skips quick-run sample folders).
!cd {ARTIFACTS} && rm -f /content/sentineldesk_artifacts.zip && zip -qr /content/sentineldesk_artifacts.zip . -x "sample_*" && ls -lh /content/sentineldesk_artifacts.zip

In [ ]:
# Option 1: Google Drive
try:
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p /content/drive/MyDrive/sentineldesk && cp /content/sentineldesk_artifacts.zip /content/drive/MyDrive/sentineldesk/
    print("Copied to My Drive/sentineldesk/sentineldesk_artifacts.zip")
except Exception as e:
    print("Drive mount not available here:", e, "-> use option 2")

In [ ]:
# Option 2: private Hugging Face model repo (artifacts derive from a CC-BY-NC dataset: keep it private).
from getpass import getpass
from huggingface_hub import HfApi
HF_REPO = "YOUR_HF_USERNAME/sentineldesk-artifacts"   # <- change
api = HfApi(token=getpass("HF write token: "))
api.create_repo(HF_REPO, repo_type="model", private=True, exist_ok=True)
api.upload_file(path_or_fileobj="/content/sentineldesk_artifacts.zip", path_in_repo="sentineldesk_artifacts.zip",
                repo_id=HF_REPO, repo_type="model")
print(f"Uploaded. On the laptop: hf download {HF_REPO} sentineldesk_artifacts.zip --local-dir .")

## On the laptop
```bash
# from the repo root, with the zip downloaded next to it
mkdir -p artifacts && unzip -o sentineldesk_artifacts.zip -d artifacts
python -c "from sentineldesk.triage.predict import Triage; print(Triage.load('artifacts').predict(None, 'VPN down for all users')['critical'])"
```